# Notebook 09: Multi-Disease Calibration Audit
#
 Systematic calibration audit across all 14 NIH pathologies.
 Runs the complete StatCalib pipeline on each disease automatically.
#
 Input:  data/scores_all_diseases.csv
 Output: data/master_results_14diseases.csv
#
 SSDI Units 1-7: all statistical methods applied to all diseases
 Novel contribution: first systematic calibration audit across
 all NIH ChestX-ray14 pathologies with formal hypothesis testing

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120
matplotlib.rcParams['font.size'] = 10
from scipy.stats import chi2, norm, wilcoxon
from sklearn.linear_model import LogisticRegression, LogisticRegressionCV
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings("ignore")
import os

In [4]:
N_BINS       = 10
N_BOOTSTRAP  = 500   # reduced for speed across 14 diseases
ALPHA        = 0.01
RANDOM_SEED  = 42
CAL_SIZE     = 0.70
np.random.seed(RANDOM_SEED)

# The 14 NIH pathologies
NIH_PATHOLOGIES = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration",
    "Mass", "Nodule", "Pneumonia", "Pneumothorax",
    "Consolidation", "Edema", "Emphysema", "Fibrosis",
    "Pleural_Thickening", "Hernia"
]

# Load all disease scores
df_all = pd.read_csv("../data/scores_all_diseases.csv")
print(f"Loaded: {len(df_all):,} rows, {len(df_all.columns)} columns")
print(f"Diseases to audit: {len(NIH_PATHOLOGIES)}")

Loaded: 10,000 rows, 29 columns
Diseases to audit: 14


In [5]:
# ── ALL STATISTICAL FUNCTIONS ─────────────────────────────────────────────────

def compute_ece(y_true, y_prob, n_bins=10):
    """Compute Expected Calibration Error."""
    bins = np.linspace(0, 1, n_bins + 1)
    ece  = 0.0
    n    = len(y_true)
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if mask.sum() == 0:
            continue
        ece += (mask.sum()/n) * abs(
            y_true[mask].mean() - y_prob[mask].mean()
        )
    return ece

def compute_mce(y_true, y_prob, n_bins=10):
    """Compute Maximum Calibration Error."""
    bins    = np.linspace(0, 1, n_bins + 1)
    max_gap = 0.0
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i+1])
        if mask.sum() < 5:
            continue
        gap = abs(y_true[mask].mean() - y_prob[mask].mean())
        if gap > max_gap:
            max_gap = gap
    return max_gap

def hosmer_lemeshow_test(y_true, y_prob, n_bins=10):
    """Hosmer-Lemeshow goodness-of-fit test."""
    quantiles  = np.percentile(y_prob, np.linspace(0, 100, n_bins+1))
    quantiles  = np.unique(quantiles)
    hl_stat    = 0.0
    valid_bins = 0
    for i in range(len(quantiles)-1):
        if i < len(quantiles)-2:
            mask = (y_prob >= quantiles[i]) & (y_prob < quantiles[i+1])
        else:
            mask = (y_prob >= quantiles[i]) & (y_prob <= quantiles[i+1])
        n_bin = mask.sum()
        if n_bin < 5:
            continue
        O_pos = y_true[mask].sum()
        E_pos = y_prob[mask].sum()
        O_neg = n_bin - O_pos
        E_neg = n_bin - E_pos
        if E_pos > 0 and E_neg > 0:
            hl_stat += (O_pos-E_pos)**2/E_pos + (O_neg-E_neg)**2/E_neg
            valid_bins += 1
    df_hl   = max(valid_bins-2, 1)
    p_value = 1 - chi2.cdf(hl_stat, df=df_hl)
    return hl_stat, p_value

def bootstrap_ece(y_true, y_prob, n_bootstrap=500, n_bins=10):
    """Bootstrap ECE distribution."""
    n      = len(y_true)
    values = np.zeros(n_bootstrap)
    for b in range(n_bootstrap):
        idx      = np.random.choice(n, size=n, replace=True)
        values[b] = compute_ece(y_true[idx], y_prob[idx], n_bins)
    return values

def fit_calibration_methods(y_cal, s_cal, y_test, s_test):
    """
    Fit all three calibration methods on calibration set.
    Evaluate on test set. Returns ECE for each method.
    """
    X_cal  = s_cal.reshape(-1, 1)
    X_test = s_test.reshape(-1, 1)

    # Platt Scaling
    try:
        platt = LogisticRegression(
            C=1e10, solver="lbfgs",
            random_state=RANDOM_SEED,
            max_iter=1000
        )
        platt.fit(X_cal, y_cal)
        s_platt   = platt.predict_proba(X_test)[:, 1]
        ece_platt = compute_ece(y_test, s_platt, N_BINS)
        auc_platt = roc_auc_score(y_test, s_platt)
    except Exception:
        ece_platt = np.nan
        auc_platt = np.nan

    # Isotonic Regression
    try:
        iso = IsotonicRegression(
            out_of_bounds="clip", increasing=True
        )
        iso.fit(s_cal, y_cal)
        s_iso   = iso.predict(s_test)
        ece_iso = compute_ece(y_test, s_iso, N_BINS)
        auc_iso = roc_auc_score(y_test, s_iso)
    except Exception:
        ece_iso = np.nan
        auc_iso = np.nan
        s_iso   = s_test

    # Ridge-Platt
    try:
        ridge = LogisticRegressionCV(
            Cs=[0.001, 0.01, 0.1, 1.0, 10.0],
            cv=3,
            penalty="l2",
            solver="lbfgs",
            scoring="neg_log_loss",
            random_state=RANDOM_SEED,
            max_iter=1000
        )
        ridge.fit(X_cal, y_cal)
        s_ridge   = ridge.predict_proba(X_test)[:, 1]
        ece_ridge = compute_ece(y_test, s_ridge, N_BINS)
        auc_ridge = roc_auc_score(y_test, s_ridge)
    except Exception:
        ece_ridge = np.nan
        auc_ridge = np.nan

    return {
        "platt":  {"ece": ece_platt, "auc": auc_platt},
        "iso":    {"ece": ece_iso,   "auc": auc_iso},
        "ridge":  {"ece": ece_ridge, "auc": auc_ridge},
        "s_iso":  s_iso
    }

print("All functions defined. Ready to run 14-disease audit loop.")

All functions defined. Ready to run 14-disease audit loop.


In [10]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.model_selection import StratifiedKFold
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, log_loss, brier_score_loss

# ── Constants ─────────────────────────────────────────────────────────────────
N_BINS        = 10
N_BOOTSTRAP   = 1000     # 300 is exploratory; 1000 gives stable 95% CIs
N_SPLITS      = 5
MIN_POSITIVES = 20       # skip disease if fewer positives than this
MIN_BIN_SIZE  = 5        # minimum samples to count a bin in ECE
RANDOM_SEED   = 42
np.random.seed(RANDOM_SEED)

NIH_PATHOLOGIES = [
    "Atelectasis", "Cardiomegaly", "Effusion", "Infiltration",
    "Mass", "Nodule", "Pneumonia", "Pneumothorax",
    "Consolidation", "Edema", "Emphysema", "Fibrosis",
    "Pleural_Thickening", "Hernia",
]

df_all = pd.read_csv("../data/scores_all_diseases.csv")
print(f"Loaded {len(df_all):,} rows, {len(df_all.columns)} columns")


# ── ECE (quantile bins) ───────────────────────────────────────────────────────

def compute_ece_quantile(y_true, y_prob, n_bins=N_BINS, min_bin_size=MIN_BIN_SIZE):
    """
    Compute Expected Calibration Error using equal-frequency (quantile) bins.

    Quantile bins ensure each bin has comparable sample counts, making ECE
    stable even when scores are concentrated in a narrow range.

    Args:
        y_true:       Binary ground-truth labels (0/1), shape (n,)
        y_prob:       Predicted probabilities in [0, 1], shape (n,)
        n_bins:       Number of bins (default 10)
        min_bin_size: Bins with fewer samples than this are skipped

    Returns:
        ece: float — expected calibration error in [0, 1]
    """
    bin_edges = np.unique(np.percentile(y_prob, np.linspace(0, 100, n_bins + 1)))
    n         = len(y_true)
    ece       = 0.0

    for i in range(len(bin_edges) - 1):
        if i < len(bin_edges) - 2:
            mask = (y_prob >= bin_edges[i]) & (y_prob < bin_edges[i + 1])
        else:
            mask = (y_prob >= bin_edges[i]) & (y_prob <= bin_edges[i + 1])

        if mask.sum() < min_bin_size:
            continue

        acc  = y_true[mask].mean()
        conf = y_prob[mask].mean()
        ece += (mask.sum() / n) * abs(acc - conf)

    return ece


# ── Cross-validated calibration ───────────────────────────────────────────────

def cross_validated_calibration(y, s, method="isotonic", n_splits=N_SPLITS):
    """
    Fit a calibration model using stratified K-fold cross-validation.

    Avoids ECE underestimation from fitting and evaluating on the same data,
    which is especially problematic for Isotonic Regression (near-zero
    training ECE but poor generalisation on rare diseases).

    Args:
        y:        Binary ground-truth labels, shape (n,)
        s:        Raw model confidence scores, shape (n,)
        method:   "isotonic" or "platt"
        n_splits: Number of CV folds (default 5)

    Returns:
        calibrated: float array of calibrated probabilities, shape (n,)
    """
    skf        = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)
    calibrated = np.empty_like(s)

    for train_idx, test_idx in skf.split(s, y):
        s_train, s_test = s[train_idx], s[test_idx]
        y_train         = y[train_idx]

        if method == "isotonic":
            clf = IsotonicRegression(out_of_bounds="clip")
            clf.fit(s_train, y_train)
            calibrated[test_idx] = clf.predict(s_test)

        elif method == "platt":
            clf = LogisticRegression(max_iter=1000)
            clf.fit(s_train.reshape(-1, 1), y_train)
            calibrated[test_idx] = clf.predict_proba(s_test.reshape(-1, 1))[:, 1]

        else:
            raise ValueError(f"Unknown method: {method!r}. Use 'isotonic' or 'platt'.")

    return calibrated


# ── Bootstrap CI ──────────────────────────────────────────────────────────────

def bootstrap_ci(y, s, metric_fn, n_bootstrap=N_BOOTSTRAP, confidence=0.95):
    """
    Compute bootstrap percentile confidence interval for a scalar metric.

    Args:
        y:           Binary ground-truth labels, shape (n,)
        s:           Predicted probabilities, shape (n,)
        metric_fn:   Callable (y, s) -> float
        n_bootstrap: Number of bootstrap replicates (default 1000)
        confidence:  CI level (default 0.95 → 2.5th/97.5th percentile)

    Returns:
        (lo, hi): lower and upper CI bounds as floats
    """
    n      = len(y)
    alpha  = (1 - confidence) / 2
    values = []

    for _ in range(n_bootstrap):
        idx = np.random.choice(n, n, replace=True)
        try:
            values.append(metric_fn(y[idx], s[idx]))
        except Exception:
            # Skip replicates where metric is undefined (e.g. all-negative sample)
            continue

    if len(values) < n_bootstrap // 2:
        # Too many failures — return NaN CI rather than mislead
        return (float("nan"), float("nan"))

    lo, hi = np.percentile(values, [alpha * 100, (1 - alpha) * 100])
    return (float(lo), float(hi))


# ── Main audit loop ───────────────────────────────────────────────────────────

results = []
skipped = []   # diseases dropped from analysis and why

print("=" * 72)
print("MULTI-DISEASE CALIBRATION AUDIT")
print("=" * 72)
print(f"{'Disease':<25} {'N+':>5} {'AUC':>6} "
      f"{'ECE_orig':>9} {'ECE_best':>9} {'Reduc%':>7} {'Method'}")
print("-" * 72)

for disease in NIH_PATHOLOGIES:

    score_col = f"score_{disease}"
    gt_col    = f"gt_{disease}"

    if score_col not in df_all.columns:
        print(f"{disease:<25} MISSING column — skipped")
        continue

    y = df_all[gt_col].values.astype(int)
    s = df_all[score_col].values.astype(float)

    n_pos = int(y.sum())
    if n_pos < MIN_POSITIVES:
        reason = f"{n_pos} positives in {len(y):,} samples — below MIN_POSITIVES={MIN_POSITIVES}"
        skipped.append({"Disease": disease, "N_pos": n_pos, "Reason": reason})
        print(f"  [SKIPPED] {disease:<25} {reason}")
        continue

    # Original metrics
    auc_orig    = roc_auc_score(y, s)
    ece_orig    = compute_ece_quantile(y, s)
    brier_orig  = brier_score_loss(y, s)
    logloss_orig = log_loss(y, s)

    # Calibrate (CV)
    s_iso   = cross_validated_calibration(y, s, method="isotonic")
    s_platt = cross_validated_calibration(y, s, method="platt")

    # Metrics after calibration
    method_preds = {"Isotonic": s_iso, "Platt": s_platt}
    metrics_after = {
        name: {
            "ECE":     compute_ece_quantile(y, preds),
            "Brier":   brier_score_loss(y, preds),
            "LogLoss": log_loss(y, preds),
            "AUC":     roc_auc_score(y, preds),
        }
        for name, preds in method_preds.items()
    }

    best_method = min(metrics_after, key=lambda x: metrics_after[x]["ECE"])
    best_preds  = method_preds[best_method]
    best        = metrics_after[best_method]

    # AUC preservation check — monotone calibrators must not change AUC
    auc_drop = auc_orig - best["AUC"]
    if auc_drop > 0.005:
        print(f"  WARNING {disease}: AUC dropped {auc_drop:.4f} after {best_method} calibration")

    # Bootstrap CIs on original and best calibrated ECE
    ci_orig = bootstrap_ci(y, s, compute_ece_quantile)
    ci_best = bootstrap_ci(y, best_preds, compute_ece_quantile)

    reduction = (1 - best["ECE"] / ece_orig) * 100

    results.append({
        "Disease":      disease,
        "N_pos":        n_pos,
        "AUC":          round(auc_orig, 4),
        "ECE_orig":     round(ece_orig, 4),
        "ECE_best":     round(best["ECE"], 4),
        "Reduction_pct": round(reduction, 1),
        "Best_Method":  best_method,
        "Brier_orig":   round(brier_orig, 4),
        "Brier_best":   round(best["Brier"], 4),
        "LogLoss_orig": round(logloss_orig, 4),
        "LogLoss_best": round(best["LogLoss"], 4),
        "AUC_after":    round(best["AUC"], 4),
        "CI_orig_lo":   round(ci_orig[0], 4),
        "CI_orig_hi":   round(ci_orig[1], 4),
        "CI_best_lo":   round(ci_best[0], 4),
        "CI_best_hi":   round(ci_best[1], 4),
    })

    print(f"{disease:<25} {n_pos:>5} {auc_orig:>6.3f} "
          f"{ece_orig:>9.4f} {best['ECE']:>9.4f} "
          f"{reduction:>6.1f}% {best_method}")

print("=" * 72)

# ── Dropped-disease log ───────────────────────────────────────────────────────
if skipped:
    print(f"\n{'='*72}")
    print(f"DISEASES DROPPED FROM ANALYSIS ({len(skipped)} of {len(NIH_PATHOLOGIES)})")
    print(f"{'='*72}")
    for s in skipped:
        print(f"  {s['Disease']:<25} {s['Reason']}")
    print(f"  → These diseases were excluded because calibration metrics")
    print(f"    are unreliable with fewer than {MIN_POSITIVES} positive samples.")
    print(f"  → Total audited: {len(results)}/{len(NIH_PATHOLOGIES)} diseases")
    print(f"{'='*72}")

# ── Summary table ─────────────────────────────────────────────────────────────

df_results = pd.DataFrame(results)

print(f"\nAudited {len(df_results)} diseases")
print(f"Avg ECE reduction: {df_results['Reduction_pct'].mean():.1f}%")
print(f"Best single reduction: {df_results['Reduction_pct'].max():.1f}% "
      f"({df_results.loc[df_results['Reduction_pct'].idxmax(), 'Disease']})")
print(f"Worst: {df_results['Reduction_pct'].min():.1f}% "
      f"({df_results.loc[df_results['Reduction_pct'].idxmin(), 'Disease']})")

# Save — use ../data/ not the notebook cwd
df_results.to_csv("../data/robust_calibration_results.csv", index=False)
print("\nSaved: ../data/robust_calibration_results.csv")

# ── ECE before/after bar chart ────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(14, 6))
x      = np.arange(len(df_results))
width  = 0.35

bars_before = ax.bar(x - width/2, df_results["ECE_orig"], width,
                     label="Before calibration", color="crimson", alpha=0.8)
bars_after  = ax.bar(x + width/2, df_results["ECE_best"], width,
                     label="After calibration (best method)", color="steelblue", alpha=0.8)

# Error bars from bootstrap CI
# clip to >= 0: bootstrap mean can sit slightly above/below the CV ECE
lo_orig = np.maximum(0, (df_results["ECE_orig"] - df_results["CI_orig_lo"]).to_numpy())
hi_orig = np.maximum(0, (df_results["CI_orig_hi"] - df_results["ECE_orig"]).to_numpy())
lo_best = np.maximum(0, (df_results["ECE_best"] - df_results["CI_best_lo"]).to_numpy())
hi_best = np.maximum(0, (df_results["CI_best_hi"] - df_results["ECE_best"]).to_numpy())

ax.errorbar(x - width/2, df_results["ECE_orig"],
            yerr=[lo_orig, hi_orig],
            fmt="none", color="black", capsize=3, linewidth=1)
ax.errorbar(x + width/2, df_results["ECE_best"],
            yerr=[lo_best, hi_best],
            fmt="none", color="black", capsize=3, linewidth=1)

ax.set_xticks(x)
ax.set_xticklabels(df_results["Disease"], rotation=40, ha="right", fontsize=9)
ax.set_ylabel("ECE (lower = better calibrated)")
ax.set_title("Multi-Disease Calibration Audit — DenseNet-121 on NIH Chest X-ray\n"
             "ECE before and after post-hoc calibration (error bars = 95% bootstrap CI)")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")
ax.set_ylim(0, df_results["ECE_orig"].max() * 1.25)
plt.tight_layout()
plt.savefig("../outputs/plots/10_multi_disease_calibration.png", dpi=150, bbox_inches="tight")
plt.close()
print("Saved: ../outputs/plots/10_multi_disease_calibration.png")


Loaded 10,000 rows, 29 columns
MULTI-DISEASE CALIBRATION AUDIT
Disease                      N+    AUC  ECE_orig  ECE_best  Reduc% Method
------------------------------------------------------------------------
  WARNING Atelectasis: AUC dropped 0.0079 after Isotonic calibration
Atelectasis                 890  0.742    0.2750    0.0082   97.0% Isotonic
  WARNING Cardiomegaly: AUC dropped 0.0081 after Isotonic calibration
Cardiomegaly                200  0.858    0.1495    0.0024   98.4% Isotonic
Effusion                    895  0.812    0.1768    0.0049   97.2% Isotonic
Infiltration               1430  0.679    0.2826    0.0048   98.3% Isotonic
  WARNING Mass: AUC dropped 0.0107 after Isotonic calibration
Mass                        309  0.760    0.2572    0.0037   98.5% Isotonic
  WARNING Nodule: AUC dropped 0.0127 after Isotonic calibration
Nodule                      452  0.648    0.3223    0.0060   98.1% Isotonic
  WARNING Pneumonia: AUC dropped 0.0207 after Platt calibration
Pneum